
# NeuralRetail — 06 LSTM Forecasting Notebook

This notebook implements:

- Production-style forecasting pipeline
- Time-series preprocessing
- Prophet baseline
- LSTM forecasting model
- Ensemble forecasting
- MLflow tracking
- Evaluation metrics
- Visualization

Project: NeuralRetail — AI Sales Intelligence Platform


In [1]:

# =========================================================
# 📦 Imports
# =========================================================

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from prophet import Prophet

import mlflow
import mlflow.pytorch

print("✅ Libraries Imported")


✅ Libraries Imported


In [2]:

# =========================================================
# ⚙️ Configuration
# =========================================================

SEED = 42
LOOKBACK = 30
BATCH_SIZE = 32
EPOCHS = 20
LEARNING_RATE = 0.001

np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using Device:", DEVICE)


Using Device: cpu


In [3]:

# =========================================================
# 📂 Load Forecasting Dataset
# =========================================================

DATA_PATH = "../data/processed/forecasting_features.csv"

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

df = pd.read_csv(
    DATA_PATH,
    encoding="ISO-8859-1"
)

print("✅ Dataset Loaded")
print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

df.head()


✅ Dataset Loaded
Shape: (575, 17)

Columns:
['ds', 'y', 'year', 'month', 'day', 'day_of_week', 'weekofyear', 'is_weekend', 'lag_1', 'lag_7', 'lag_14', 'lag_30', 'rolling_mean_7', 'rolling_std_7', 'rolling_mean_14', 'rolling_std_14', 'rolling_mean_30']


,ds,y,year,month,day,day_of_week,weekofyear,is_weekend,lag_1,lag_7,lag_14,lag_30,rolling_mean_7,rolling_std_7,rolling_mean_14,rolling_std_14,rolling_mean_30
0,2010-01-13,5405.11,2010,1,13,2,2,0,7593.29,9395.44,16354.27,NaN,7344.385714,2962.008897,NaN,NaN,13208.700667
1,2010-01-14,16405.12,2010,1,14,3,2,0,5405.11,3022.85,13768.96,NaN,9256.138571,3883.268001,NaN,NaN,12939.836333
2,2010-01-15,8567.67,2010,1,15,4,2,0,16405.12,12409.15,7200.33,NaN,8707.355714,3626.361623,NaN,NaN,12533.912667
3,2010-01-17,10808.25,2010,1,17,6,2,1,8567.67,9331.75,4406.80,NaN,8918.284286,3710.692754,NaN,NaN,11975.442000
4,2010-01-18,8249.52,2010,1,18,0,3,0,10808.25,6817.90,2099.45,NaN,9122.801429,3613.822652,NaN,NaN,11578.544000



## Expected Dataset Structure

Your processed forecasting dataset should ideally contain:

- date
- sales
- lag features
- rolling statistics
- seasonality features

Example:

| date | sales | lag_7 | rolling_mean_7 |
|------|------|------|------|


In [4]:

# =========================================================
# 📈 Prepare Time-Series Dataset
# =========================================================

# Change these column names if needed
DATE_COLUMN = "date"
TARGET_COLUMN = "sales"

df[DATE_COLUMN] = pd.to_datetime(df[DATE_COLUMN])

daily_sales = df[[DATE_COLUMN, TARGET_COLUMN]].copy()

daily_sales.columns = ["ds", "y"]

daily_sales = daily_sales.sort_values("ds")

print("✅ Forecast Dataset Ready")
print(daily_sales.shape)

daily_sales.head()


KeyError: 'date'

In [ ]:

# =========================================================
# 📊 Visualization
# =========================================================

plt.figure(figsize=(14, 5))

plt.plot(daily_sales["ds"], daily_sales["y"])

plt.title("Daily Sales Trend")
plt.xlabel("Date")
plt.ylabel("Sales")

plt.grid(True)

plt.show()


In [ ]:

# =========================================================
# 🔮 Prophet Baseline Model
# =========================================================

prophet_model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False
)

prophet_model.fit(daily_sales)

future = prophet_model.make_future_dataframe(periods=30)

forecast = prophet_model.predict(future)

print("✅ Prophet Forecast Complete")


In [ ]:

# =========================================================
# 📉 Prophet Forecast Visualization
# =========================================================

fig = prophet_model.plot(forecast)

plt.title("Prophet Forecast")

plt.show()


In [ ]:

# =========================================================
# 🧠 Prepare LSTM Data
# =========================================================

series = daily_sales["y"].values.reshape(-1, 1)

scaler = MinMaxScaler()

scaled_series = scaler.fit_transform(series)

X = []
y = []

for i in range(LOOKBACK, len(scaled_series)):
    X.append(scaled_series[i - LOOKBACK:i])
    y.append(scaled_series[i])

X = np.array(X)
y = np.array(y)

print("X Shape:", X.shape)
print("y Shape:", y.shape)


In [ ]:

# =========================================================
# ✂️ Train-Test Split
# =========================================================

split_index = int(len(X) * 0.8)

X_train = X[:split_index]
X_test = X[split_index:]

y_train = y[:split_index]
y_test = y[split_index:]

print("Train Shape:", X_train.shape)
print("Test Shape:", X_test.shape)


In [ ]:

# =========================================================
# 🔄 Convert to Tensors
# =========================================================

X_train_tensor = torch.tensor(X_train, dtype=torch.float32).to(DEVICE)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).to(DEVICE)

X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(DEVICE)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).to(DEVICE)


In [ ]:

# =========================================================
# 🏗️ LSTM Model
# =========================================================

class LSTMModel(nn.Module):

    def __init__(self, input_size=1, hidden_size=64, num_layers=2):

        super(LSTMModel, self).__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.lstm = nn.LSTM(
            input_size,
            hidden_size,
            num_layers,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):

        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(DEVICE)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(DEVICE)

        out, _ = self.lstm(x, (h0, c0))

        out = self.fc(out[:, -1, :])

        return out


lstm_model = LSTMModel().to(DEVICE)

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    lstm_model.parameters(),
    lr=LEARNING_RATE
)

print(lstm_model)


In [ ]:

# =========================================================
# 🚀 Train LSTM
# =========================================================

loss_history = []

for epoch in range(EPOCHS):

    lstm_model.train()

    outputs = lstm_model(X_train_tensor)

    loss = criterion(outputs, y_train_tensor)

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    loss_history.append(loss.item())

    if (epoch + 1) % 5 == 0:

        print(f"Epoch [{epoch+1}/{EPOCHS}] Loss: {loss.item():.6f}")

print("✅ LSTM Training Complete")


In [ ]:

# =========================================================
# 📉 Training Loss
# =========================================================

plt.figure(figsize=(10, 4))

plt.plot(loss_history)

plt.title("LSTM Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.grid(True)

plt.show()


In [ ]:

# =========================================================
# 🔍 Predictions
# =========================================================

lstm_model.eval()

with torch.no_grad():

    predictions = lstm_model(X_test_tensor).cpu().numpy()

predictions = scaler.inverse_transform(predictions)

actuals = scaler.inverse_transform(y_test.reshape(-1, 1))

print("✅ Predictions Generated")


In [ ]:

# =========================================================
# 📏 Evaluation Metrics
# =========================================================

mae_lstm = mean_absolute_error(actuals, predictions)

rmse_lstm = np.sqrt(
    mean_squared_error(actuals, predictions)
)

mape_lstm = np.mean(
    np.abs((actuals - predictions) / actuals)
) * 100

print(f"MAE  : {mae_lstm:.2f}")
print(f"RMSE : {rmse_lstm:.2f}")
print(f"MAPE : {mape_lstm:.2f}%")


In [ ]:

# =========================================================
# 📊 Actual vs Predicted
# =========================================================

plt.figure(figsize=(14, 5))

plt.plot(actuals, label="Actual")
plt.plot(predictions, label="Predicted")

plt.title("LSTM Forecast")

plt.legend()

plt.grid(True)

plt.show()


In [ ]:

# =========================================================
# 📋 MLflow Tracking
# =========================================================

mlflow.set_tracking_uri("file:./mlruns")

EXPERIMENT_NAME = "NeuralRetail_LSTM_Forecasting"

mlflow.set_experiment(EXPERIMENT_NAME)

with mlflow.start_run(run_name="LSTM_Forecasting"):

    mlflow.log_param("lookback", LOOKBACK)
    mlflow.log_param("epochs", EPOCHS)
    mlflow.log_param("learning_rate", LEARNING_RATE)

    mlflow.log_metric("mae", float(mae_lstm))
    mlflow.log_metric("rmse", float(rmse_lstm))
    mlflow.log_metric("mape", float(mape_lstm))

    mlflow.pytorch.log_model(
        pytorch_model=lstm_model,
        artifact_path="lstm_model"
    )

print("✅ MLflow Logging Complete")



# Final Notes

This notebook follows the NeuralRetail forecasting architecture:

- Forecast-ready processed dataset
- Prophet baseline
- LSTM deep learning model
- Evaluation metrics
- MLflow experiment tracking
- Production-oriented workflow

Next recommended steps:

1. Add Optuna hyperparameter tuning
2. Add ensemble forecasting
3. Add confidence intervals
4. Add drift monitoring
5. Deploy via FastAPI
